# CNN track — train & evaluate (farmyard SED)

2D CNN over a 64-bin log-mel spectrogram, multi-label sigmoid head (5 animals +
background). Trained on 1-second windows **synthesised on the fly** from the
processed fold 1–4 clips (0/1/2 animals mixed onto a background bed at random SNR),
so the training distribution matches continuous, overlapping inference audio.
Validated on the fold-5 hold-out (no leakage) and scored with the shared event
harness (±500 ms collar).

**Reported result:** event macro-F1 **0.371**, segment macro-F1 0.47.

Runtime: **GPU** (Runtime → Change runtime type → T4), then Run All. Trains from the
code already in `src/` — no setup cells, no patching.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys

REPO_DIR = '/content/drive/MyDrive/ASP-GRP4-ANIMAL-SOUND-DETECTION'   # change if you renamed the folder
MAX_EPOCHS      = 80
PATIENCE        = 12
N_TUNE_SCENES   = 8      # threshold sweep scenes (seeds 100+)
N_EVAL_SCENES   = 8      # reported scenes (seeds 0+, disjoint from tuning)
CKPT_PATH       = os.path.join(REPO_DIR, 'cnn_sed.ckpt.pt')   # best-so-far, survives a disconnect
RESUME          = os.path.isfile(CKPT_PATH)

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
os.chdir(REPO_DIR)
assert os.path.isfile('processed/farmyard.csv'), 'REPO_DIR is wrong: no processed/farmyard.csv'

import torch
print('GPU:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(enable a GPU runtime!)')
print('resume from checkpoint:', RESUME)

import torchaudio, glob
try:
    torchaudio.load(glob.glob('processed/cat/*.wav')[0]); print('audio backend OK')
except Exception:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchcodec'], check=False)
    torchaudio.load(glob.glob('processed/cat/*.wav')[0]); print('audio backend OK after install')

## 1. Overfit sanity check (skip on a resume)

In [ ]:
if not RESUME:
    from cnn_model import overfit_check
    score = overfit_check(n_windows=32, epochs=80)
    print(f'overfit-check train macro-F1: {score:.3f}  (expect ~1.0 -> the model can learn)')
else:
    score = float('nan'); print('resuming -> skipping overfit check')

## 2. Train to convergence (early stopping, checkpointed to Drive)
Best weights are saved to `cnn_sed.ckpt.pt` on every validation improvement — if
Colab disconnects, just Run All again and it warm-starts from the checkpoint.

In [ ]:
from cnn_model import train_model, save_model

result = train_model(max_epochs=MAX_EPOCHS, patience=PATIENCE,
                     checkpoint_path=CKPT_PATH,
                     resume_from=CKPT_PATH if RESUME else None,
                     verbose=True)
model = result['model']
save_model(model, os.path.join(REPO_DIR, 'cnn_sed.pt'))
print(f"\nbest val frame macro-F1: {result['val_f1']:.3f} at epoch {result['best_epoch']}")
print('saved -> cnn_sed.pt')

## 3. Learning curves

In [ ]:
import matplotlib.pyplot as plt
h = result['history']
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(h['train_loss'], label='train'); a1.plot(h['val_loss'], label='val')
a1.set_xlabel('epoch'); a1.set_title('loss'); a1.legend()
a2.plot(h['val_f1'], color='green'); a2.set_xlabel('epoch'); a2.set_title('val macro-F1 (animals)')
fig.tight_layout(); fig.savefig(os.path.join(REPO_DIR, 'cnn_training_curves.png')); plt.show()

## 4. Per-class frame-level P/R/F1 on fold-5 (cheap diagnostic, before event formation)

In [ ]:
from cnn_model import frame_level_report
report = frame_level_report(model)
print(f"{'class':12} {'P':>6} {'R':>6} {'F1':>6} {'support':>8}")
for label, r in report.items():
    if label == 'macro_animals': continue
    print(f"{label:12} {r['precision']:6.3f} {r['recall']:6.3f} {r['f1']:6.3f} {r['support']:8d}")
print(f"\nanimal macro-F1: {report['macro_animals']['f1']:.3f}")

## 5. Visualize predictions on a held-out scene (qualitative)

In [ ]:
from cnn_model import plot_predictions
from scene_synthesis import synthesize_scene, load_holdout_frames
holdout = load_holdout_frames()
scene_wav, scene_gt = synthesize_scene(holdout, seed=0)
path = plot_predictions(model, scene_wav, 16000, out_path=os.path.join(REPO_DIR, 'cnn_predictions.png'))
print('ground truth:', [(e.label, round(e.onset, 1), round(e.offset, 1)) for e in scene_gt])
from IPython.display import Image, display
display(Image(path))

## 6. Event-based evaluation (±500 ms) via the shared harness
Per-class thresholds are tuned on held-out scenes (seeds 100+) and the score is
reported on **disjoint** scenes (seeds 0+), so it isn't inflated by tuning.

In [ ]:
import numpy as np
from cnn_model import predict_fn_for, predict_timeline
from scene_synthesis import synthesize_scene, evaluate_on_scenes, tune_thresholds, load_holdout_frames
from postprocessing import timeline_to_events
from evaluation import event_based_metrics, ANIMAL_CLASSES

config = tune_thresholds(predict_fn_for(model), n_scenes=N_TUNE_SCENES, seed_start=100)
print('tuned per-class thresholds:', {k: round(v, 2) for k, v in config.per_class_threshold_on.items()})
print('tuned min_duration:', config.min_duration_seconds)

summary = evaluate_on_scenes(predict_fn_for(model), n_scenes=N_EVAL_SCENES, config=config, seed_start=0)
print()
print('event   macro-F1 (held-out scenes):', round(summary['event_macro_f1'], 3))
print('segment macro-F1:', round(summary['segment_macro_f1'], 3))
print('near-miss:', summary['near_miss'])

holdout = load_holdout_frames()
acc = {c: {'precision': [], 'recall': [], 'f1': []} for c in ANIMAL_CLASSES}
for i in range(N_EVAL_SCENES):
    wav, gt = synthesize_scene(holdout, seed=i)
    prob, times, energy = predict_timeline(model, wav, 16000)
    pred, _ = timeline_to_events(prob, times, config, energy_db=energy)
    m = event_based_metrics(pred, gt)
    for c in ANIMAL_CLASSES:
        for k in ('precision', 'recall', 'f1'):
            acc[c][k].append(m[c][k])
print()
print(f"{'class':10} {'P':>6} {'R':>6} {'F1':>6}")
for c in ANIMAL_CLASSES:
    print(f"{c:10} {np.mean(acc[c]['precision']):6.3f} {np.mean(acc[c]['recall']):6.3f} {np.mean(acc[c]['f1']):6.3f}")

## 7. Save the metrics report

In [ ]:
report_path = os.path.join(REPO_DIR, 'cnn_model_results.txt')
with open(report_path, 'w') as f:
    f.write('CNN model (2D conv over mel) - George\n')
    f.write(f'overfit-check macro-F1: {score:.3f}\n')
    f.write(f"best val frame macro-F1: {result['val_f1']:.3f} (epoch {result['best_epoch']})\n")
    f.write(f'tuned per-class thresholds: {config.per_class_threshold_on}\n')
    f.write(f'tuned min_duration: {config.min_duration_seconds}\n')
    f.write(f"event   macro-F1 ({N_EVAL_SCENES} held-out scenes): {summary['event_macro_f1']:.3f}\n")
    f.write(f"segment macro-F1: {summary['segment_macro_f1']:.3f}\n")
    f.write(f"near-miss: {summary['near_miss']}\n\n")
    f.write(f"{'class':10} {'P':>6} {'R':>6} {'F1':>6}\n")
    for c in ANIMAL_CLASSES:
        f.write(f"{c:10} {np.mean(acc[c]['precision']):6.3f} {np.mean(acc[c]['recall']):6.3f} {np.mean(acc[c]['f1']):6.3f}\n")
    f.write('\nframe-level per-class (fold-5):\n')
    for label, r in report.items():
        if label != 'macro_animals':
            f.write(f"{label:10} P{r['precision']:.3f} R{r['recall']:.3f} F1{r['f1']:.3f}\n")
print('saved ->', report_path)
print(open(report_path).read())